In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd() if (Path.cwd() / "s2_pore_phase.py").exists() else Path.cwd() / "evaluation" / "digital_core"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import s2_pore_phase as s2


# =========================
# 3D XYZ directional S2 plot
# =========================

OUTPUT_DIR = Path("S2_results_3D")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_R = min(127, min(s2.DIMENSION) - 1)
DEVICE = "auto"  # auto / cpu / gpu

GROUPS = {
    "TI": s2.TESTB_FOLDER,
    "DCGAN": s2.DCB_FOLDER,
    "STGAN": s2.ST_FOLDER,
    "StyleGAN": s2.STYLE_FOLDER,
    "ours": s2.GEB_FOLDER,
}

DIRECTIONS = {
    "X": ("x", 0),
    "Y": ("y", 1),
    "Z": ("z", 2),
}

# 3D line styles. Keep colors synced with s2_pore_phase.py, but use simple line styles
# because 3D projection can make dense dash patterns harder to read.
GROUP_3D_STYLES = {
    "TI": {"color": "#1F2933", "linewidth": 3.2, "linestyle": "-", "alpha": 1.0},
    "DCGAN": {"color": "#4C78A8", "linewidth": 2.1, "linestyle": "--", "alpha": 0.92},
    "STGAN": {"color": "#54A24B", "linewidth": 2.1, "linestyle": "-.", "alpha": 0.92},
    "StyleGAN": {"color": "#F2A541", "linewidth": 2.1, "linestyle": ":", "alpha": 0.95},
    "ours": {"color": "#D94E67", "linewidth": 2.8, "linestyle": "-", "alpha": 0.98},
}

DIRECTION_ANGLES = {
    "X": 0.0,
    "Z": 45.0,
    "Y": 90.0,
}

GROUP_ANGLE_OFFSETS = {
    "TI": -3.0,
    "DCGAN": -1.5,
    "STGAN": 0.0,
    "StyleGAN": 1.5,
    "ours": 3.0,
}

PLOT_RC = {
    "font.family": "Arial",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 15,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


def calculate_xyz_groups():
    backend = s2.resolve_device(DEVICE)
    print(f"Device: {s2.backend_label(backend)}")
    print("Only X/Y/Z directional S2 is calculated; radial FFT is skipped.")

    curves = {}
    for group_name, folder in GROUPS.items():
        curves[group_name] = s2.calculate_group(
            group_name=group_name,
            folder=folder,
            max_r=MAX_R,
            max_files=None,
            do_radial=False,
            backend=backend,
        )
    return curves


def set_3d_pane_style(ax):
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((0.97, 0.97, 0.97, 1.0))
        axis.pane.set_edgecolor((0.82, 0.82, 0.82, 1.0))
    ax.grid(False)


def fan_direction_points(direction_name, r, s2_values, angle_offset=0.0):
    theta = np.deg2rad(DIRECTION_ANGLES[direction_name] + angle_offset)
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    z = np.asarray(s2_values, dtype=float)
    return x, y, z


def draw_fan_guides(ax):
    for direction_name, angle in DIRECTION_ANGLES.items():
        theta = np.deg2rad(angle)
        x_end = MAX_R * np.cos(theta)
        y_end = MAX_R * np.sin(theta)
        ax.plot([0, x_end], [0, y_end], [0, 0], color="#9AA5B1", linewidth=1.0, alpha=0.65)
        ax.text(x_end * 1.04, y_end * 1.04, 0, direction_name, color="#222222", fontsize=11)


def plot_xyz_3d(group_curves, output_dir=OUTPUT_DIR):
    r = np.arange(MAX_R + 1)

    with plt.rc_context(PLOT_RC):
        fig = plt.figure(figsize=(11.5, 7.4), dpi=150)
        ax = fig.add_subplot(111, projection="3d")

        for group_name, curves in group_curves.items():
            style = GROUP_3D_STYLES[group_name]
            angle_offset = GROUP_ANGLE_OFFSETS[group_name]
            for direction_name, (curve_key, direction_pos) in DIRECTIONS.items():
                label = group_name if direction_name == "X" else None
                x, y, z = fan_direction_points(direction_name, r, curves[curve_key], angle_offset)
                ax.plot(
                    x,
                    y,
                    z,
                    label=label,
                    **style,
                )

        draw_fan_guides(ax)

        ax.set_title("Directional pore-phase $S_2(r)$ fan plot", pad=18)
        ax.set_xlabel("X-plane lag projection (voxel)", labelpad=12)
        ax.set_ylabel("Y-plane lag projection (voxel)", labelpad=12)
        ax.set_zlabel("$S_2(r)$", labelpad=10)

        axis_pad = 8
        all_s2 = np.concatenate([
            curves[key] for curves in group_curves.values() for key in ("x", "y", "z")
        ])
        z_max = float(np.nanmax(all_s2))
        ax.set_xlim(-axis_pad, MAX_R + axis_pad)
        ax.set_ylim(-axis_pad, MAX_R + axis_pad)
        ax.set_zlim(0, z_max * 1.08)
        xy_ticks = [0, MAX_R // 2, MAX_R]
        z_ticks = np.linspace(0, z_max, 4)
        ax.set_xticks(xy_ticks)
        ax.set_yticks(xy_ticks)
        ax.set_zticks(z_ticks)
        ax.set_zticklabels([f"{tick:.2f}" for tick in z_ticks])
        ax.set_box_aspect((1, 1, 0.62))

        ax.view_init(elev=28, azim=-54)
        ax.legend(loc="upper left", bbox_to_anchor=(0.02, 0.95), frameon=True, framealpha=0.88)
        set_3d_pane_style(ax)

        fig.tight_layout()
        png_path = output_dir / "S2_xyz_fan_3D.png"
        pdf_path = output_dir / "S2_xyz_fan_3D.pdf"
        fig.savefig(png_path, dpi=600)
        fig.savefig(pdf_path, dpi=600)
        plt.show()

    print(f"Saved: {png_path.resolve()}")
    print(f"Saved: {pdf_path.resolve()}")


group_curves = calculate_xyz_groups()
plot_xyz_3d(group_curves)
